<a href="https://colab.research.google.com/github/MaganJadhav/Gen-AI/blob/main/Word2vec_2__by_MJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.6 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import re

import gensim
from gensim.models import Word2Vec

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
# prompt: load data from google drive https://drive.google.com/file/d/1vZ4S0dtiUk5LqeeccqWs9IAAG8qH1GWv/view?usp=sharing

!gdown --id 1vZ4S0dtiUk5LqeeccqWs9IAAG8qH1GWv
!unzip News_Category_Dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1vZ4S0dtiUk5LqeeccqWs9IAAG8qH1GWv
From (redirected): https://drive.google.com/uc?id=1vZ4S0dtiUk5LqeeccqWs9IAAG8qH1GWv&confirm=t&uuid=c0a1e2e6-5c0a-4ed4-9ce0-48cbb03389e2
To: /content/News_Category_Dataset.zip
100% 27.8M/27.8M [00:00<00:00, 39.0MB/s]
Archive:  News_Category_Dataset.zip
  inflating: News_Category_Dataset_v3.json  


In [7]:
# Step 1: Load the Dataset
data = pd.read_json('/content/News_Category_Dataset_v3.json', lines=True)
data

# We'll use the 'headline' as the text data and 'category' as the label
data = data[['headline', 'category']].dropna()

data['processed_text'] = data['headline'].astype(str)
print(data['category'].value_counts())

#Consider top4 categories only
top_categories = ['POLITICS', 'ENTERTAINMENT', 'BUSINESS', 'SPORTS']
data = data[data['category'].isin(top_categories)]
print(data["headline"].head())
print(data['category'].value_counts())

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [8]:
## Step 3: Prepare the Data for TensorFlow

# Initialize the Tokenizer to convert text into sequences of integers.
tokenizer = Tokenizer()

# Fit the tokenizer on the 'processed_text' column to learn the vocabulary.
tokenizer.fit_on_texts(data['processed_text'])

# Convert the text in 'processed_text' column to sequences of integers.
sequences = tokenizer.texts_to_sequences(data['processed_text'])

# Define the maximum length for the sequences. Any sequences longer than this will be truncated,
# and any sequences shorter will be padded.
max_length = 100  # Maximum length of a complaint narrative

# Pad or truncate the sequences so that they all have the same length of max_length.
X = pad_sequences(sequences, maxlen=max_length)

# Convert the 'category' column to numerical values using LabelEncoder.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['category'])

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (51226, 100)
Shape of y_train: (51226,)
Shape of X_test: (12807, 100)
Shape of y_test: (12807,)


In [9]:
## Step 4: Configure the model

# Calculate the vocabulary size. The vocabulary size is the total number of unique words in the text data plus one.
vocab_size = len(tokenizer.word_index) + 1  # Vocabulary size
print(vocab_size)

# Initialize a Sequential model.
model = Sequential()

# Add an Embedding layer. This layer will learn word embeddings for the input sequences.
# - input_dim: Size of the vocabulary.
# - output_dim: Dimension of the dense embedding.
# - input_length: Length of input sequences.
model.add(Embedding(input_dim=vocab_size, output_dim=32))
# Add a GlobalAveragePooling1D layer. This layer calculates the average of all the embeddings in a sequence.
# This reduces the dimensionality and helps to prevent overfitting.
model.add(GlobalAveragePooling1D())

# Add a Dense layer with 64 units and ReLU activation. This layer acts as a hidden layer in the neural network.
model.add(Dense(64, activation='relu'))

# Add a Dense output layer with a number of units equal to the number of unique labels.
# The softmax activation function is used to output a probability distribution over the classes.
model.add(Dense(len(label_encoder.classes_), activation='softmax'))
model.summary()

41828


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
## Step 5: Train the Model

# Compile the model. The compile step specifies the optimizer, loss function, and evaluation metrics.
# - optimizer: 'adam' is a popular optimizer that adjusts the learning rate during training.
# - loss: 'sparse_categorical_crossentropy' is used for multi-class classification when labels are provided as integers.
# - metrics: 'accuracy' will track the accuracy of the model during training and evaluation.
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model using the training data.
# - X_train: Training input data.
# - y_train: Training target data.
# - epochs: Number of times to iterate over the training data.
# - batch_size: Number of samples per gradient update.
# - validation_data: Data on which to evaluate the loss and any model metrics at the end of each epoch.
model.fit(X_train, y_train, epochs=5, batch_size=128, validation_data=(X_test, y_test))

# Save the trained model's weights to a file.
model.save_weights('complaints_model.weights.h5')

# Load the model weights from the saved file. This step can be used to reload the model for further evaluation or inference.
model.load_weights('complaints_model.weights.h5')

Epoch 1/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.5576 - loss: 1.0950 - val_accuracy: 0.5497 - val_loss: 1.0738
Epoch 2/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - accuracy: 0.6662 - loss: 0.8961 - val_accuracy: 0.7134 - val_loss: 0.7975
Epoch 3/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 14s 35ms/step - accuracy: 0.7638 - loss: 0.6806 - val_accuracy: 0.7686 - val_loss: 0.6514
Epoch 4/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.7775 - loss: 0.5959 - val_accuracy: 0.7247 - val_loss: 0.6969
Epoch 5/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.7982 - loss: 0.5196 - val_accuracy: 0.7975 - val_loss: 0.5325


In [11]:
## Step 6: Evaluate the Model

# Predict the classes for the test data.
# The model.predict method returns probabilities for each class, and np.argmax is used to get the class with the highest probability.
y_pred = np.argmax(model.predict(X_test), axis=1)

# Calculate the confusion matrix to evaluate the accuracy of the classification.
# The confusion matrix shows the number of correct and incorrect predictions for each class.
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Calculate the accuracy score by comparing the true labels with the predicted labels.
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Print the classification report, which includes precision, recall, f1-score, and support for each class.
print("Classification Report:")
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
print(report)

401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
[[ 406  359  525    1]
 [  11 3242  217    0]
 [  40  450 6550    0]
 [  42  843  105   16]]
Accuracy: 0.7975325993597252
Classification Report:
               precision    recall  f1-score   support

     BUSINESS       0.81      0.31      0.45      1291
ENTERTAINMENT       0.66      0.93      0.78      3470
     POLITICS       0.89      0.93      0.91      7040
       SPORTS       0.94      0.02      0.03      1006

     accuracy                           0.80     12807
    macro avg       0.83      0.55      0.54     12807
 weighted avg       0.82      0.80      0.76     12807



In [12]:
# Making a prediction on new narrations

# Define a list of new complaint texts to predict their categories.
news_new = [
    """
    LOS ANGELES -- With the bases loaded and two outs against one of baseball’s
    nastiest relievers, MJ Melendez fought off pitch after pitch … after pitch after pitch … to
    keep the at-bat alive in hopes of coming through in the Royals’
    best scoring opportunity on Saturday night.
    """,
    """
    Biden campaign rakes in $28 million for star-studded Los Angeles fundraiser
    The massive haul was announced just hours before President Joe Biden appeared
    alongside former President Barack Obama, George Clooney and others.
    """
]

# Convert the new complaint texts into sequences of integers using the previously fitted tokenizer.
new_sequences = tokenizer.texts_to_sequences(news_new)

# Pad the sequences so that they all have the same length as the training data (max_length).
new_X = pad_sequences(new_sequences, maxlen=max_length)

# Predict the class probabilities for the new complaint sequences.
new_predictions = model.predict(new_X)

# Determine the predicted class for each new complaint by finding the index of the maximum probability.
pred_class = np.argmax(new_predictions, axis=1)

# Print the predicted class indices.
print(pred_class)

# Convert the predicted class indices back to class labels using the label encoder.
pred_labels = label_encoder.inverse_transform(pred_class)

# Print the predicted class labels.
print(pred_labels)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
[1 2]
['ENTERTAINMENT' 'POLITICS']
